# Demo — Asesor de viajes multi-agente (entrega final)

Este notebook usa la **API real** levantada con `docker compose up` (app + Redis 8), como lo haría cualquier
cliente. Muestra, en este orden:

1. `GET /health`: Redis, checkpointer, RAG y LangSmith.
2. `GET /rag/buscar`: el retriever híbrido solo (BM25 + vectorial + RRF + re-ranking), sin LLM.
3. Una pregunta **estable** que se resuelve solo con las guías (sin pausa ni web).
4. Un pedido con presupuesto: **pausa human-in-the-loop** → approve → respuesta con fuentes.
5. Un **seguimiento** en el mismo `thread_id` ("¿y si somos 4?") y el historial con `GET /threads/{id}`.

Cada job queda trazado en LangSmith (proyecto `entrega-final`, metadata `job_id` / `thread_id` / `turno`).

In [1]:
import json, time
import httpx
from IPython.display import Markdown, display

API = "http://localhost:8000"
c = httpx.Client(base_url=API, timeout=30)

def esperar(job_id, estados=("DONE", "FAILED", "WAITING_APPROVAL"), limite=600):
    fin = time.time() + limite
    while time.time() < fin:
        job = c.get(f"/tasks/{job_id}").json()
        if job["estado"] in estados:
            return job
        time.sleep(2)
    raise TimeoutError(job_id)

def resumen(job):
    r = job.get("resultado") or {}
    m = r.get("metricas") or {}
    return {"estado": job["estado"], "turno": job["turno"], "tramos": job["tramos"], "ruta": r.get("ruta"),
            "permiso_web": r.get("permiso_web"),
            "fuentes (guía/memoria/web)": f"{m.get('fuentes_guia')}/{m.get('fuentes_memoria')}/{m.get('fuentes_web')}",
            "búsquedas web": m.get("busquedas_web"), "tiempo de grafo (s)": job.get("tiempo_ejecucion_s"),
            "tokens": job.get("total_tokens"), "costo (US$)": job.get("costo_usd")}

## 1. Salud del servicio

In [2]:
h = c.get("/health").json()
print(json.dumps({k: h[k] for k in ("status", "modo", "modelo", "redis", "checkpointer", "langsmith")}, indent=2, ensure_ascii=False))
print(json.dumps({k: h["rag"][k] for k in ("store", "fragmentos", "embeddings", "reranker", "destinos")}, indent=2, ensure_ascii=False))

{
  "status": "ok",
  "modo": "real",
  "modelo": "claude-haiku-5-5",
  "redis": {
    "ok": true,
    "tipo": "redis"
  },
  "checkpointer": "AsyncRedisSaver",
  "langsmith": {
    "tracing": true,
    "proyecto": "entrega-final",
    "clave": true
  }
}
{
  "store": "chroma:indice_chroma/guias_viaje",
  "fragmentos": 246,
  "embeddings": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "reranker": "jinaai/jina-reranker-v2-base-multilingual",
  "destinos": [
    "Bariloche",
    "Córdoba",
    "El Calafate",
    "Iguazú",
    "Mar del Plata",
    "Mendoza",
    "Puerto Madryn",
    "Salta",
    "Ushuaia"
  ]
}


## 2. El RAG solo: `GET /rag/buscar` (filtro por destino desde los metadatos)

In [3]:
r = c.get("/rag/buscar", params={"q": "qué hacer en invierno, esquí", "destino": "Ushuaia", "k": 3}).json()
print("guía:", r["slug"], "| candidatos vectoriales:", r["candidatos_vectoriales"], "| BM25:", r["candidatos_bm25"],
      "| re-ranker:", r["reranker"], "|", r["milisegundos"], "ms")
for x in r["fragmentos"]:
    f = x["fragmento"]
    print(f"\n[{f['id']}] § {f['seccion']}  (puesto vectorial {x['puesto_vectorial']}, BM25 {x['puesto_bm25']}, "
          f"RRF {x['puntaje_rrf']:.4f}, re-rank {x['puntaje_rerank']})")
    print("   ", f["texto"][:220].replace("\n", " "), "...")

guía: ushuaia | candidatos vectoriales: 20 | BM25: 7 | re-ranker: jinaai/jina-reranker-v2-base-multilingual | 3120 ms

[guia:ushuaia#017] § Hacer > Invierno  (puesto vectorial 1, BM25 1, RRF 0.0328, re-rank 0.4998)
    En invierno, el área se convierte en una gran estación de esquí. En Ushuaia hay dos pistas en el Glaciar Martial, ambas con vista al mar.  - **Cerro Castor**: Si buscas más, este centro de deportes de invierno ofrece esq ...

[guia:ushuaia#014] § Hacer  (puesto vectorial 2, BM25 2, RRF 0.0323, re-rank 0.2939)
    Es una ciudad que tiene encanto en cada temporada. En invierno, el clima es muy frío, pero es la mejor temporada para practicar esquí y otros deportes de montaña, como el trineo con perros siberianos y el motonieve. El l ...

[guia:ushuaia#018] § Comprar  (puesto vectorial None, BM25 6, RRF 0.0152, re-rank -1.7876)
    Cerca de la costanera y principalmente en la **avenida San Martín** se encuentra la principal zona comercial y al oeste de la ciudad y al norte d

## 3. Una pregunta estable: solo guías, sin pausa ni búsquedas web

In [4]:
t = c.post("/tasks", json={"pregunta": "¿Cómo llego del aeropuerto al centro de Mendoza y qué bodegas puedo visitar?"}).json()
job = esperar(t["job_id"])
print(json.dumps(resumen(job), indent=2, ensure_ascii=False))
display(Markdown(job["resultado"]["respuesta_final"]))

{
  "estado": "DONE",
  "turno": 1,
  "tramos": 1,
  "ruta": "supervisor → aprobacion_web → investigador → supervisor → sintesis",
  "permiso_web": "sin_pedir",
  "fuentes (guía/memoria/web)": "9/0/0",
  "búsquedas web": 0,
  "tiempo de grafo (s)": 29.83,
  "tokens": 37568,
  "costo (US$)": 0.005467
}


## Cómo llegar del aeropuerto al centro

El aeropuerto es el Gobernador Francisco Gabrielli (El Plumerillo, MDZ), a unos 11 km del centro [guia:mendoza#002]. Tenés dos opciones:

- **Remis o taxi:** tienen precios fijos publicados. La guía cita AR$225 al centro, pero es un dato de 2019, así que tomalo solo como referencia [guia:mendoza#002].
- **Colectivo urbano:** pasa cada 40 minutos y la guía indica unos 60 minutos hasta el centro. Hay una inconsistencia en la fuente, que en un pasaje dice 2 horas [guia:mendoza#002]. Necesitás tarjeta de autobús, que no se vende en el aeropuerto, así que conviene llevarla desde antes.

Si llegás por micro de larga distancia, la terminal está a unos 2 km al este del centro. Desde ahí hay taxis y remis, y a pie son unos 15 minutos, aunque la guía no lo recomienda de noche [guia:mendoza#003].

## Bodegas para visitar

- **Zonas principales:** Maipú y Luján de Cuyo son las más conocidas por el Malbec. También destacan el Cabernet Sauvignon y el Merlot [guia:mendoza#013].
- **Bodegas con visitas sin cita:** Norton y Rutini, entre otras [guia:mendoza#007].
- **Consejo práctico:** muchas bodegas cierran los fines de semana y conviene reservar con antelación [guia:mendoza#007].
- **Época ideal:** la cosecha, en marzo y abril [guia:mendoza#007].
- **Alojamiento cerca de bodegas:** Posada Borravino (Chacras de Coria, cerca de bodegas muy lindas), Club Tapiz (Luján de Cuyo), Cavas Wine Lodge (fuera de la ciudad) y Tupungato Divino (Tupungato) [guia:mendoza#017].
- **Tours de degustación:** Potrerillos Explorer ofrece catas y también trekking, rafting y cabalgatas [guia:mendoza#009].

## Otras actividades cercanas

- **Parque San Martín:** sendero desde el zoológico hasta el Cerro de la Gloria, con buena vista al atardecer. Se puede alquilar bici en "Bicis del Parque" [guia:mendoza#007].
- **Museos:** Museo Municipal de Arte Moderno, Museo del Área Fundacional y Museo Provincial de Bellas Artes (Casa Fader) [guia:mendoza#008].
- **Termas de Cacheuta:** a una hora de la ciudad, con el Parque del Agua. Se llega en bus desde la terminal (Espreso Uspallata) [guia:mendoza#009].

## Lo que no pudimos averiguar

- Los nombres completos de las bodegas, sus horarios, precios de visitas y rutas por zona. Para armar un recorrido de bodegas hacen falta fuentes más específicas.
- Los precios actuales de remis, colectivo y Cacheuta. Las guías traen valores viejos.
- El mes del viaje, que define si la visita a bodegas cae en temporada de cosecha. Si me decís cuándo viajás, puedo ajustar las recomendaciones.

## Fuentes
- [guía] Wikivoyage, «Mendoza» § Llegar > Por avión — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Llegar > Por bus — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Ver — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Dormir > Derroche — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Ver > Museos — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Hacer — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)
- [guía] Wikivoyage, «Mendoza» § Comer y Beber — https://es.wikivoyage.org/wiki/Mendoza (CC BY-SA 4.0)

_La información de destinos (qué hacer, cómo llegar, clima típico) sale de las guías de Wikivoyage en español, bajo licencia CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/deed.es)._

## 4. Pedido con presupuesto: pausa human-in-the-loop antes de gastar en la web

In [5]:
t = c.post("/tasks", json={"pregunta": "Quiero ir 4 días a Puerto Madryn en septiembre para ver ballenas, somos 2, "
                                        "salimos de Buenos Aires y tenemos 1.200.000 pesos. ¿Nos alcanza?"}).json()
pausa = esperar(t["job_id"])
print("estado:", pausa["estado"])
print(json.dumps(pausa.get("aprobacion_pendiente"), indent=2, ensure_ascii=False))

estado: WAITING_APPROVAL
{
  "tipo": "aprobacion_busqueda_web",
  "ronda": 1,
  "destino": "Puerto Madryn",
  "temas_sin_memoria": [
    "pasajes",
    "alojamiento",
    "comida"
  ],
  "temas_en_memoria": [
    "cambio"
  ],
  "temas_en_guias": [
    "actividades",
    "clima",
    "como_llegar"
  ],
  "consultas_previstas": [
    "precio de pasajes Buenos Aires → Puerto Madryn",
    "precio de alojamiento por noche en Puerto Madryn",
    "cuánto cuesta comer en Puerto Madryn"
  ],
  "instruccion_del_supervisor": "Buscá datos para un viaje de Buenos Aires a Puerto Madryn, ida 10/09/2027 y vuelta 13/09/2027, 2 personas, 4 días, presupuesto total 1.200.000 ARS. Temas: pasajes (precios de referencia de vuelos o micro Buenos Aires–Puerto Madryn para 2 personas, ida y vuelta, en septiembre; rangos o valores 'desde' están bien), alojamiento (precio de referencia por noche para 2 personas en Puerto Madryn en septiembre), comida (precio de referencia por día), actividades (avistaje de ballen

In [6]:
if pausa["estado"] == "WAITING_APPROVAL":
    print(c.post(f"/tasks/{t['job_id']}/approve", json={"aprobado": True, "comentario": "demo del notebook"}).json())
job = esperar(t["job_id"], ("DONE", "FAILED"))
print(json.dumps(resumen(job), indent=2, ensure_ascii=False))
display(Markdown(job["resultado"]["respuesta_final"]))
thread_id = job["thread_id"]

{'job_id': 'c3bb855350bf4b249efe7dd27da157ec', 'estado': 'RUNNING', 'aprobado': True}


{
  "estado": "DONE",
  "turno": 1,
  "tramos": 2,
  "ruta": "supervisor → aprobacion_web → investigador → supervisor → aprobacion_web → investigador → supervisor → analista → supervisor → sintesis",
  "permiso_web": "aprobada",
  "fuentes (guía/memoria/web)": "9/12/9",
  "búsquedas web": 3,
  "tiempo de grafo (s)": 83.03,
  "tokens": 130187,
  "costo (US$)": 0.01959
}


## Respuesta

No podemos confirmar si el viaje entra en $1.200.000. Los datos que tenemos no permiten calcular el presupuesto con precios confiables en pesos.

**Viaje validado:** Buenos Aires → Puerto Madryn, del 10/09/2027 al 13/09/2027 (4 días, 3 noches), 2 personas. Supusimos esas fechas porque pediste "septiembre" sin día exacto; si preferís otra fecha, avisanos.

## Itinerario

No armamos itinerario día por día porque sin precios no sabemos qué combinación de actividades y alojamiento es viable en el presupuesto. Lo que sí sabemos:

- **Avistaje de ballenas:** entre mayo y diciembre llega la ballena franca austral para reproducirse [guia:puerto-madryn#000]. Los barcos salen de Puerto Pirámides, en la Península Valdés [guia:puerto-madryn#007]. Ojo: Puerto Pirámides queda a una distancia considerable de Madryn, así que conviene tenerlo en cuenta para la logística.
- **Clima:** no tenemos datos de septiembre. La guía indica temperaturas bajas en invierno (promedio de 12 °C), sin nieve, y vientos muy intensos en esa época [guia:puerto-madryn#002].
- **Otras opciones:** Punta Loma (lobos marinos) y buceo en el Golfo Nuevo [guia:puerto-madryn#007]. Punta Tombo (pingüinos) se visita de octubre a marzo, no en septiembre [guia:puerto-madryn#006].

## Presupuesto

No pudimos calcularlo. Estos son los renglones y por qué:

- **Pasajes:** las fuentes dan precios en UAH (Google Flights [mem-a77910cb693b]) o sin moneda clara (Expedia "$291" y "$237" [mem-ba08bd28cd71]; Skyscanner "desde $113" [mem-0fe4922e8859]). No podemos usarlos como ARS.
- **Alojamiento:** KAYAK muestra "$36", "$40", "$45" y "$49" por noche sin moneda ni aclarar si es por habitación o por persona [mem-2227095dbfb3]. Otra página de KAYAK da "$132.564/noche" [mem-93e5c25f555d], probablemente en pesos colombianos. Ninguna cifra sirve como ARS confirmado.
- **Comida:** el único dato en ARS es "5 viandas por $40.000" [mem-ad0d068f9d8f], unos $8.000 por vianda. No es un costo diario por persona. El "$44.400 por día" de Facebook [mem-4ce25e8f1ba9] no indica cantidad de personas ni fecha.
- **Cambio:** las cotizaciones de memoria son de 2026-11-25 y no coinciden entre fuentes. Oficial entre $1.579 y $1.615; MEP entre $1.572 y $1.693 [mem-71caa0983bd4], [mem-118a59bdea75], [mem-d041e707b47c].

Estimar con estos valores sería inventar números, así que no lo hicimos.

## Lo que no pudimos averiguar

- Precio del pasaje Buenos Aires–Puerto Madryn en ARS por persona, para septiembre de 2027.
- Precio de alojamiento por noche en ARS, aclarando si es por habitación o por persona.
- Costo de comida por persona por día.
- Precio de excursiones de avistaje de ballenas (es opcional, no bloquea el cálculo).
- Precio actual del micro Buenos Aires–Puerto Madryn. La guía indica unas 17 horas de viaje [guia:puerto-madryn#003], pero no el costo.
- Cotización del dólar vigente, con fuente y fecha.

**Próximo paso:** con una búsqueda específica de estos precios en ARS, podemos correr el cálculo. Si el resultado no alcanza, evaluamos opciones como micro en lugar de avión o alojamiento compartido.

## Fuentes
- [guía] Wikivoyage, «Puerto Madryn» § Hacer — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Comprender — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Ver — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Comprender > Clima — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Llegar — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [memoria] mem-71caa0983bd4 → https://www.infobae.com/economia/divisas/dolar-hoy — cambio, buscado el 2026-11-25
- [memoria] mem-118a59bdea75 → https://www.lanacion.com.ar/dolar-hoy — cambio, buscado el 2026-11-25
- [memoria] mem-d041e707b47c → https://www.dolarito.ar — cambio, buscado el 2026-11-25
- [memoria] mem-ba08bd28cd71 → https://www.expedia.com/es/lp/vuelos/aep/pmy/buenos-aires-a-puerto-madryn — pasajes, buscado el 2026-11-25
- [memoria] mem-0fe4922e8859 → https://www.skyscanner.com/us/es-mx/usd/rutas/buea/pmy/buenos-aires-a-puerto-madryn.html — pasajes, buscado el 2026-11-25
- [memoria] mem-a77910cb693b → https://www.google.com/travel/flights?tfs=CBwQARoeag0IAxIJL20vMDFseTVtcg0IAxIJL20vMDRicGZ6Gh5qDQgDEgkvbS8wNGJwZnpyDQgDEgkvbS8wMWx5NW1AAUgBcAGCAQsI____________AZgBAQ&tfu=KgIIAw&hl=es — pasajes, buscado el 2026-11-25
- [memoria] mem-2227095dbfb3 → https://www.es.kayak.com/Hoteles-en-Puerto-Madryn.5374.hotel.ksp — alojamiento, buscado el 2026-11-25
- [memoria] mem-93e5c25f555d → https://www.kayak.com.co/Hoteles-en-Puerto-Madryn.5374.hotel.ksp — alojamiento, buscado el 2026-11-25
- [memoria] mem-4ce25e8f1ba9 → https://www.facebook.com/groups/724633018682127/posts/737466967398732 — comida, buscado el 2026-11-25
- [memoria] mem-ad0d068f9d8f → https://www.facebook.com/masenmadryn/posts/-organiz%C3%A1-tus-comidas-y-ahorr%C3%A1-todos-los-d%C3%ADas-vas-a-comer-rico-y-variado-y-como-/1561478349343106 — comida, buscado el 2026-11-25

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

_La información de destinos (qué hacer, cómo llegar, clima típico) sale de las guías de Wikivoyage en español, bajo licencia CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/deed.es)._

## 5. Seguimiento en la misma conversación (`thread_id`) e historial

In [7]:
t2 = c.post("/tasks", json={"pregunta": "¿Y si en vez de 2 somos 4?", "thread_id": thread_id}).json()
print("turno:", t2["turno"], "| thread:", t2["thread_id"] == thread_id)
job2 = esperar(t2["job_id"])
if job2["estado"] == "WAITING_APPROVAL":
    c.post(f"/tasks/{t2['job_id']}/approve", json={"aprobado": True})
    job2 = esperar(t2["job_id"], ("DONE", "FAILED"))
print(json.dumps(resumen(job2), indent=2, ensure_ascii=False))
print("pedido del turno 2:", json.dumps(job2["resultado"]["pedido"], ensure_ascii=False))
display(Markdown(job2["resultado"]["respuesta_final"]))

turno: 2 | thread: True


{
  "estado": "DONE",
  "turno": 2,
  "tramos": 1,
  "ruta": "supervisor → aprobacion_web → investigador → supervisor → analista → supervisor → aprobacion_web → investigador → supervisor → sintesis",
  "permiso_web": "sin_pedir",
  "fuentes (guía/memoria/web)": "4/12/0",
  "búsquedas web": 0,
  "tiempo de grafo (s)": 62.03,
  "tokens": 102742,
  "costo (US$)": 0.015552
}
pedido del turno 2: {"origen": "Buenos Aires", "destino": "Puerto Madryn", "fecha_ida": "2027-09-10", "fecha_vuelta": "2027-09-13", "dias": 4, "personas": 4, "presupuesto": 1200000.0, "moneda": "ARS", "supuestos": ["septiembre sin día exacto: ida el 10/09/2027 (septiembre de 2026 ya pasó)", "vuelta calculada como ida + 3 días (4 días de viaje)"]}


## Respuesta

No podemos confirmar si con 4 personas el viaje entra en $1.200.000. Con los datos que tenemos, cualquier cifra sería una suposición.

**Qué cambia con 4 personas:** el pedido ahora es para 4 viajeros, a diferencia de los 2 del turno anterior. Los pasajes, el alojamiento y la comida se duplican, así que el costo total se acerca al doble, pero no podemos calcularlo porque ningún precio confirmado está en pesos argentinos.

## Presupuesto

**No calculado.** Faltan precios confiables para los tres rubros principales:

- **Pasajes:** Expedia muestra "$237" y "$291" sin moneda ni aclaración de si son por persona [mem-ba08bd28cd71]. Skyscanner dice "desde $113", probablemente en USD, sin confirmarlo [mem-0fe4922e8859]. Google Flights está en UAH y no sirve [mem-a77910cb693b].
- **Alojamiento:** KAYAK da "$45" de promedio en septiembre y "desde $49" por noche, sin moneda ni aclarar si es por habitación o por persona [mem-2227095dbfb3] [mem-54d9e60f3af6]. El precio de Colombia ("$132.564") no sirve [mem-93e5c25f555d].
- **Comida:** no hay un costo diario por persona confiable en ARS. Las cifras de Facebook son de otro contexto o de un particular [mem-4ce25e8f1ba9] [mem-ad0d068f9d8f].

**Tipo de cambio (referencia):** oficial entre $1.579 y $1.615; tarjeta entre $2.052 y $2.099; MEP entre $1.572 y $1.693 por USD [mem-71caa0983bd4] [mem-118a59bdea75] [mem-d041e707b47c]. Conviene confirmar la moneda de los precios antes de elegir cuál usar.

## Itinerario

No armamos itinerario día por día. Sin presupuesto confirmado no sabemos qué combinación de actividades y alojamiento es viable.

Lo que sí sabemos sobre las actividades: en septiembre se puede hacer avistaje de ballenas (ballena franca austral) desde Puerto Pirámides, en la Península Valdés [guia:puerto-madryn#000] [guia:puerto-madryn#007]. También hay snorkel con lobos marinos, buceo en el Golfo Nuevo y Punta Loma, a 17 km [guia:puerto-madryn#007]. Punta Tombo no aplica en septiembre porque se visita de octubre a marzo [guia:puerto-madryn#006]. Para el clima, la guía indica temperaturas bajas (promedio de 12 °C) y vientos muy intensos en esa época [guia:puerto-madryn#002].

## Lo que no pudimos averiguar

- Precio de pasajes ida y vuelta en ARS, por persona.
- Precio por noche en ARS, con unidad clara (habitación o persona).
- Costo diario de comida por persona en ARS.
- Precio actual del avistaje de ballenas. El único dato citado es de 2019.

Para completar el cálculo hace falta una búsqueda web específica por cada rubro, pidiendo precios en pesos argentinos. Requiere autorización del supervisor.

## Fuentes
- [guía] Wikivoyage, «Puerto Madryn» § Hacer — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Comprender — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [guía] Wikivoyage, «Puerto Madryn» § Ver — https://es.wikivoyage.org/wiki/Puerto_Madryn (CC BY-SA 4.0)
- [memoria] mem-ba08bd28cd71 → https://www.expedia.com/es/lp/vuelos/aep/pmy/buenos-aires-a-puerto-madryn — pasajes, buscado el 2026-11-25
- [memoria] mem-0fe4922e8859 → https://www.skyscanner.com/us/es-mx/usd/rutas/buea/pmy/buenos-aires-a-puerto-madryn.html — pasajes, buscado el 2026-11-25
- [memoria] mem-a77910cb693b → https://www.google.com/travel/flights?tfs=CBwQARoeag0IAxIJL20vMDFseTVtcg0IAxIJL20vMDRicGZ6Gh5qDQgDEgkvbS8wNGJwZnpyDQgDEgkvbS8wMWx5NW1AAUgBcAGCAQsI____________AZgBAQ&tfu=KgIIAw&hl=es — pasajes, buscado el 2026-11-25
- [memoria] mem-2227095dbfb3 → https://www.es.kayak.com/Hoteles-en-Puerto-Madryn.5374.hotel.ksp — alojamiento, buscado el 2026-11-25
- [memoria] mem-54d9e60f3af6 → https://www.kayak.com.sv/Hoteles-en-Puerto-Madryn.5374.hotel.ksp — alojamiento, buscado el 2026-11-25
- [memoria] mem-93e5c25f555d → https://www.kayak.com.co/Hoteles-en-Puerto-Madryn.5374.hotel.ksp — alojamiento, buscado el 2026-11-25
- [memoria] mem-4ce25e8f1ba9 → https://www.facebook.com/groups/724633018682127/posts/737466967398732 — comida, buscado el 2026-11-25
- [memoria] mem-ad0d068f9d8f → https://www.facebook.com/masenmadryn/posts/-organiz%C3%A1-tus-comidas-y-ahorr%C3%A1-todos-los-d%C3%ADas-vas-a-comer-rico-y-variado-y-como-/1561478349343106 — comida, buscado el 2026-11-25
- [memoria] mem-71caa0983bd4 → https://www.infobae.com/economia/divisas/dolar-hoy — cambio, buscado el 2026-11-25
- [memoria] mem-118a59bdea75 → https://www.lanacion.com.ar/dolar-hoy — cambio, buscado el 2026-11-25
- [memoria] mem-d041e707b47c → https://www.dolarito.ar — cambio, buscado el 2026-11-25

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

_La información de destinos (qué hacer, cómo llegar, clima típico) sale de las guías de Wikivoyage en español, bajo licencia CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/deed.es)._

In [8]:
conv = c.get(f"/threads/{thread_id}").json()
print(conv["turnos_completos"], "turnos ·", conv["checkpointer"], "· jobs:", conv["jobs"])
for t in conv["turnos"]:
    print(f"  turno {t['turno']}: {t['pregunta']}  →  {t['pedido'].get('personas')} personas, {t['pedido'].get('destino')}")

2 turnos · AsyncRedisSaver · jobs: ['c3bb855350bf4b249efe7dd27da157ec', 'e0b9fc89f38b478aa45e0660af5e24ab']
  turno 1: Quiero ir 4 días a Puerto Madryn en septiembre para ver ballenas, somos 2, salimos de Buenos Aires y tenemos 1.200.000 pesos. ¿Nos alcanza?  →  2 personas, Puerto Madryn
  turno 2: ¿Y si en vez de 2 somos 4?  →  4 personas, Puerto Madryn
